# Does SIGReg select content by pixel mass?

A5 (`AAC/notebooks/runpod_A5.ipynb`) trained the Craftax model with no prediction loss at all and still lost the mobs, ores and other
objects in view, in the same steps as B2 and V3. The prediction loss (the race) is not what removes them; SIGReg (with the IDM) is
the suspect. This notebook tests that in the synthetic clip world, where every factor is labelled and the content's weight in the pixels
is a knob (`a_fast`, `a_hard`).

**What Craftax says (`AAC/reports/pixel_mass/pixel_mass.md`, 151k validation frames).** Each class's pixel mass is the share of the frame's
pixel variance it explains. `kept` = the readout R² of a class at step 2000 over its R² at step 0 (an untrained encoder), B2 (A5 and V3 alike).

| class | share of the pixel variance | kept at step 2000 |
|---|---|---|
| grass, sand, water, stone, path | 13.1%, 11.2%, 6.3%, 3.6%, 1.4% | 1.02, 1.03, 1.04, 1.03, 1.03 |
| tree, lava | 1.6%, 0.95% | 0.52, 0.64 |
| cow, coal, furnace | 0.34%, 0.30%, 0.26% | 0.06, 0.35, 0.05 |
| iron, zombie, crafting table | 0.17%, 0.15%, 0.15% | 0.20, 0.16, 0.26 |
| skeleton, diamond, arrow | 0.06%, 0.05%, 0.01% | 0.12, 0.10, 0.15 |
| plant (the exception) | 0.07% | 0.91 |

The player's facing sprite is 0.19%, the inventory strip 25% of the variance. Spearman (mass against kept, 18 classes): 0.73 (A5), 0.68 (B2),
0.67 (V3). Nothing above about 1.4% was lost; nothing below 0.35% was kept, apart from the plant, which `place_plant` (an action) makes. The
clip world's sprites carry 7.7% each at the default brightness, 20 to 50 times a Craftax mob or ore.

**Hypotheses, with the predictions written before any result**

| | statement | prediction in the clip world |
|---|---|---|
| **H1** | a SIGReg-shaped latent keeps content in proportion to its pixel mass, with a threshold | with `detached` (SIGReg only), r2_fast and r2_hard rise with the sprite's share of the variance; below about 0.5% they are lost (R² < 0.2), above a few percent kept (R² ≥ 0.5) |
| **H2** | the prediction loss is not the selector | `sg` (prediction + SIGReg) is within 0.15 of `detached` at every mass |
| **H3** | SIGReg is blind to time; asking it for the differences changes the selection | SIGReg on z_t − z_{t−1} keeps the sprites at low mass (R² up by 0.3 at 0.14 to 0.6% mass), raises noise, and in the differences-only arm slow content falls |
| **H4** (optional) | where the fix of the clip world (B, `retain_open`) stops working | B holds the sprites down to some mass, below which it fails like the baselines |
| **H5** | less SIGReg, less pruning (grid world: the random map, facing, hand, mob) | in B's shadow (reconstruction + SIGReg) the facing and hand readouts rise as the SIGReg weight falls; at weight 0 (a plain autoencoder) they are highest. In `sg` (prediction + SIGReg) the latent first gains labelled content as SIGReg falls, then collapses (rank falls) |

**Why there is no λ sweep.** With the SIGReg-only encoder (`detached`), SIGReg is the only loss, and AdamW is invariant to the scale of the
loss: a weight of 0.03 or 3 gives the same updates. A weight matters only against another loss (the prediction loss, the IDM), which is a
different experiment. H5 (stage 6) is that experiment: the SIGReg weight against a reconstruction loss and against the prediction loss.

Stages: 1 calibration · 2 the mass sweep (H1, H2) · 3 crowding by slow content · 4 SIGReg on the differences (H3) · 5 B on the grid (H4, optional) ·
6 the SIGReg weight in the grid world (H5) · 7 more seeds (optional) · 8 verdicts. One seed per stage; every run is resumable.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from gaf.env import ClipEnv, EnvCfg, GROUPS
from gaf.train import Cfg
from gaf import analysis as A

OUT = "/workspace/results_sigreg"     # separate from the other notebooks' results
SEED = 0
EXTRA_SEEDS = (1, 2)                  # the optional cell at the end
WORKERS = 3                           # runs at once on the GPU

# ---- the knobs ---------------------------------------------------------------------------
MASS_A = (0.5, 0.7, 1.0, 1.4, 2.0, 4.0)   # sprite brightness a_fast = a_hard (default 4: 7.7% of the pixel variance each)
STEPS = 5000
EARLY = (0, 50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000)
COMMON = dict(D=32, batch=256, lr=1e-3, lam=0.1, wd=0.01, basis="excess", floor_steps=1500, floor_warm=True)
pd.set_option("display.precision", 3)


def env(a=4.0, **kw):
    return EnvCfg(a_fast=a, a_hard=a, **kw)


def cfg(arm, seed=SEED, **kw):
    return Cfg(arm=arm, seed=seed, **{**COMMON, **kw})

## 1 Calibration: how much of the pixel variance a sprite carries
The share of the pixel variance (per-pixel variance over clips, mean over pixels) that one sprite explains: the squared error of a frame
without that sprite, over the variance. The fast sprite is white, the hard one red, so their shares differ a little. Craftax's reference
values are in the table above: mobs and ores 0.05 to 0.35%, lava 0.95%, tree 1.6%, terrain 1.4 to 13%.

In [ ]:
def sprite_share(a_fast, a_hard, which, n=2048, **kw):
    torch.manual_seed(0)
    x = ClipEnv(EnvCfg(a_fast=a_fast, a_hard=a_hard, **kw), "cpu").sample(n).x
    ref = dict(fast=dict(a_fast=0.0, a_hard=a_hard), hard=dict(a_fast=a_fast, a_hard=0.0))[which]
    torch.manual_seed(0)
    x0 = ClipEnv(EnvCfg(**ref, **kw), "cpu").sample(n).x
    return 100 * ((x - x0) ** 2).mean().item() / x.var(0).mean().item()


MASS = {a: sprite_share(a, a, "fast") for a in MASS_A}
MASS_HARD = {a: sprite_share(a, a, "hard") for a in MASS_A}
display(pd.DataFrame({"a": MASS_A, "fast sprite, % of the pixel variance": [round(MASS[a], 2) for a in MASS_A],
                      "hard sprite": [round(MASS_HARD[a], 2) for a in MASS_A]}).set_index("a"))

## 2 The mass sweep (H1, H2)
`detached` (the encoder gets only SIGReg; the predictor trains on detached latents) and `sg` (stop-gradient target, the baseline), at six
sprite brightnesses, 5000 steps each, the clip world's other settings unchanged (D = 32, 16 slow + 8 noise patterns). 12 runs.
Read `r2_fast` and `r2_hard` (ridge from z_t) with the MLP probe alongside; the early evals give the course over the first 3000 steps.

In [ ]:
E1 = [cfg(arm, steps=STEPS, early_eval=EARLY, env=env(a), tag=f"mass-a{a}") for arm in ("detached", "sg") for a in MASS_A]
A.execute(E1, OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
METRICS = ["r2_fast", "r2_hard", "mlp_r2_fast", "mlp_r2_hard", "r2_slow", "r2_noise"]


def sweep_tables(df, prefix, seeds=(SEED,), metrics=METRICS):
    d = df[df.tag.str.startswith(prefix) & df.seed.isin(seeds)].copy()
    d["mass %"] = d["env.a_fast"].map(MASS).round(2)
    return {m: d.pivot_table(index="mass %", columns="arm", values=m) for m in metrics}


def plot_mass(tabs, groups=("r2_fast", "r2_hard")):
    fig, axes = plt.subplots(1, len(groups), figsize=(4.4 * len(groups), 3.5), sharey=True, layout="constrained")
    for ax, m in zip(np.atleast_1d(axes), groups):
        for arm in tabs[m].columns:
            ax.plot(tabs[m].index, tabs[m][arm], marker="o", color=A.COLOR[arm], label=arm)
        ax.axvspan(0.05, 0.35, color="0.9", zorder=0)
        ax.axvline(1.4, ls="--", lw=1, color="0.5")
        ax.set(xscale="log", title=m.replace("r2_", ""), xlabel="sprite share of the pixel variance (%)", ylim=(-0.05, 1.02))
    np.atleast_1d(axes)[0].set_ylabel("ridge R² from z_t")
    np.atleast_1d(axes)[-1].legend()
    np.atleast_1d(axes)[0].text(0.06, 0.92, "Craftax mobs, ores", fontsize=8, color="0.4")
    np.atleast_1d(axes)[0].text(1.5, 0.05, "Craftax: kept above", fontsize=8, color="0.4")
    return fig


tabs = sweep_tables(df, "mass-a")
for m, t in tabs.items():
    print(m)
    display(t.round(3))
plot_mass(tabs);

In [ ]:
R_E1 = {f"a={a} ({MASS[a]:.2f}%)": dict(arm="detached", tag=f"mass-a{a}", seed=SEED) for a in (0.5, 1.0, 2.0, 4.0)}
A.trace(df, OUT, R_E1, cols=("r2_fast", "r2_hard", "r2_slow", "r2_noise"));

**Reading stage 2**
- r2_fast and r2_hard rise with the share and are near zero at the Craftax masses (a = 0.5 to 0.7): H1. Note where they cross 0.5; compare with
  Craftax (kept above about 1.4%). If the crossing is far above Craftax's, the clip world's loss is harsher than Craftax's for the same mass,
  and Craftax's loss has a second cause (frequency, slowness).
- `sg` close to `detached` at every mass: H2, the prediction loss is not the selector. If `sg` is clearly lower at some mass, the prediction
  loss adds to the loss there (the race, as in the clip world at the default brightness).
- Both low at every mass, even at 7.7% (a = 4): mass is not enough here; the earlier clip-world finding (fast 0.33 in `detached`) already
  points that way.
- The MLP probe higher than ridge: the sprites are held, but not linearly.

## 3 Crowding: does slow content take the room?
SIGReg normalises every direction, so the absolute pixel mass should matter less than the mass of the *other* content. Sprites at a fixed
brightness (a = 2, 2.2% of the variance in the default world) with the slow patterns' amplitude `a_slow` raised or lowered (default 1.0). The
sprites' share of the variance changes with it (printed). 3 runs of `detached` (a_slow = 1 is the stage-2 run, a = 2).

In [ ]:
E2 = [cfg("detached", steps=STEPS, early_eval=EARLY, env=env(2.0, a_slow=s), tag=f"crowd-s{s}") for s in (0.25, 4.0, 8.0)]
A.execute(E2, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for s, tag in [(1.0, "mass-a2.0"), (0.25, "crowd-s0.25"), (4.0, "crowd-s4.0"), (8.0, "crowd-s8.0")]:
    r = A.select(df, arm="detached", tag=tag, seed=SEED)
    if len(r):
        share = sprite_share(2.0, 2.0, "fast", a_slow=s)
        rows[f"a_slow={s}"] = {"fast sprite %": share, **{m: r.iloc[0][m] for m in ("r2_fast", "r2_hard", "r2_slow", "r2_noise")}}
display(pd.DataFrame(rows).T.round(3))

**Reading stage 3.** r2_fast and r2_hard falling as a_slow rises: slow content crowds the sprites out in proportion to its mass (relative mass
decides). No change: only the sprites' own mass matters.

## 4 SIGReg on the differences (H3)
`lam_diff` adds SIGReg on z_t − z_{t−1} for every pair of consecutive frames. Static content has a zero difference and cannot satisfy it;
changing content can. Three arms of `detached` at the two lowest brightnesses (a = 0.5 and 1.0): the frames only (the stage-2 run), frames and
differences (`lam = lam_diff = 0.1`), differences only (`lam = 0`, `lam_diff = 0.1`). 4 new runs.
Prediction: the sprites come back at low mass, noise rises (it changes every frame), and slow content falls in the differences-only arm.
The risk this tests: static content is no longer asked for at all.

In [ ]:
E3 = []
for a in (0.5, 1.0):
    E3 += [cfg("detached", steps=STEPS, early_eval=EARLY, env=env(a), lam_diff=0.1, tag=f"delta-a{a}-both"),
           cfg("detached", steps=STEPS, early_eval=EARLY, env=env(a), lam=0.0, lam_diff=0.1, tag=f"delta-a{a}-only")]
A.execute(E3, OUT, workers=WORKERS)
df = A.collect(OUT)
rows = {}
for a in (0.5, 1.0):
    for lab, tag in (("frames", f"mass-a{a}"), ("frames + differences", f"delta-a{a}-both"), ("differences only", f"delta-a{a}-only")):
        r = A.select(df, arm="detached", tag=tag, seed=SEED)
        if len(r):
            rows[(f"a={a} ({MASS[a]:.2f}%)", lab)] = {m: r.iloc[0][m] for m in
                                                       ("r2_fast", "r2_hard", "r2_slow", "r2_noise", "mlp_r2_fast", "mlp_r2_hard", "rank_z", "sigreg")}
t3 = pd.DataFrame(rows).T
display(t3.round(3))

**Reading stage 4**
- Sprites up in "frames + differences" and "differences only": SIGReg's blindness to time is part of the cause; the differences-term is a
  candidate fix at the regulariser (for Craftax: mobs change, terrain does not).
- Noise up as well: the changing noise patterns are the cheapest Gaussian increments. If noise takes the room the sprites were meant to get, the
  differences term needs a guard (the open path of the gate, or the shadow's predictability weights).
- Slow down in "differences only", as predicted: the state is no longer constrained. "Frames + differences" keeps both.
- Nothing changes: SIGReg's blindness to time is not the cause at these masses; the selection is by mass.

## 5 B on the mass grid (H4, optional)
`retain_open` (the reconstruction shadow with retention and the open target path), 12 000 steps, the default settings of the clip world, at
four brightnesses. Where does it stop holding the sprites? About 20 minutes per run alone on a pod (it trains a second encoder and a decoder),
4 runs. Set `RUN_B = True` to run it. The grid world suggests it fails when the content is a percent of the pixels or less.

In [ ]:
RUN_B = False
if RUN_B:
    E4 = [cfg("retain_open", steps=12_000, early_eval=EARLY, env=env(a), tag=f"mass-B-a{a}") for a in (0.5, 1.0, 2.0, 4.0)]
    A.execute(E4, OUT, workers=WORKERS)
    df = A.collect(OUT)
    rows = {}
    for a in (0.5, 1.0, 2.0, 4.0):
        r = A.select(df, arm="retain_open", tag=f"mass-B-a{a}", seed=SEED)
        if len(r):
            rows[f"a={a} ({MASS[a]:.2f}%)"] = {m: r.iloc[0][m] for m in ("r2_fast", "r2_hard", "r2_slow", "r2_noise", "sh_r2_fast", "sh_r2_hard")}
    display(pd.DataFrame(rows).T.round(3))
    display(A.drift(df, OUT, {f"a={a}": dict(arm="retain_open", tag=f"mass-B-a{a}", seed=SEED) for a in (0.5, 1.0, 2.0, 4.0)}))

## 6 The SIGReg weight in the grid world (H5)
Does the latent prune less when SIGReg is weaker? The grid world is the better ground: it has Craftax's dominant, unlabelled content (the random
map), labelled facing, hand and mob, and B's shadow, whose failure we measured there (facing and hand ≈ 0 with reconstruction and SIGReg 0.1).
The weight only means something against another loss, so two tests (D = 32, 5000 steps, the random map; weights 0, 0.01, 0.03, 0.1, 0.3, 1):

- **6a the shadow of `retain_open`** (reconstruction + SIGReg + a shadow predictor): read what the shadow holds, `sh_r2_*` (ridge) and
  `sh_mlp_r2_*` (MLP probe). The live encoder of these runs is not the point. At weight 0 the shadow is a plain autoencoder.
  If facing and hand rise as the weight falls: SIGReg prunes them. If they stay ≈ 0 even at weight 0: the reconstruction's pixel-mass weighting
  (facing 3.5%, hand 1% of the loss) is the limit, and SIGReg is not the cause of B's failure here.
- **6b the baseline `sg`** (prediction + SIGReg): at small weights the prediction loss alone collapses the latent (rank falls), so read `rank_z`
  with the content. What the latent is made of comes from `latent_determined_by` (the map, inventory + position, facing + hand, mob + twinkle).
6 + 6 runs, 5000 steps; the shadow runs take about 5 minutes alone each.

In [ ]:
from gaf.env_grid import GridCfg
GRID = GridCfg()                                  # the grid_benchmark.ipynb setting: random map, D = 32
LAMS = (0.0, 0.01, 0.03, 0.1, 0.3, 1.0)


def gcfg(arm, **kw):
    return Cfg(arm=arm, seed=SEED, **{**COMMON, "env": GRID, **kw})


E5A = [gcfg("retain_open", lam=l, steps=STEPS, early_eval=EARLY, tag=f"lam-sh{l}") for l in LAMS]
E5B = [gcfg("sg", lam=l, steps=STEPS, early_eval=EARLY, tag=f"lam-sg{l}") for l in LAMS]
A.execute(E5A + E5B, OUT, workers=WORKERS)
df = A.collect(OUT)

In [ ]:
def lam_table(df, prefix, arm, cols):
    rows = {}
    for l in LAMS:
        r = A.select(df, arm=arm, tag=f"{prefix}{l}", seed=SEED)
        if len(r):
            rows[l] = {c: r.iloc[0].get(c, np.nan) for c in cols}
    return pd.DataFrame(rows).T.rename_axis("SIGReg weight")


SH = ["sh_r2_fast", "sh_mlp_r2_fast", "sh_r2_hard", "sh_mlp_r2_hard", "sh_r2_slow", "sh_r2_noise", "sh_mlp_r2_noise", "shadow_rec_mse"]
t6a = lam_table(df, "lam-sh", "retain_open", SH)
print("6a: what the shadow holds (reconstruction + SIGReg at the given weight)")
display(t6a.round(3))
fig, ax = plt.subplots(figsize=(5.2, 3.4), layout="constrained")
for c, ls in (("sh_mlp_r2_fast", "-"), ("sh_mlp_r2_hard", "-"), ("sh_r2_fast", ":"), ("sh_r2_hard", ":")):
    if c in t6a:
        ax.plot(t6a.index, t6a[c], ls=ls, marker="o", label=c.replace("sh_", ""), color=A.GROUP_COLOR["fast" if "fast" in c else "hard"])
ax.set(xscale="symlog", xlabel="SIGReg weight (0 = plain autoencoder)", ylabel="what the shadow holds (R²)", ylim=(-0.05, 1.02))
ax.legend(fontsize=8);

In [ ]:
print("6b: the baseline sg (prediction + SIGReg)")
t6b = lam_table(df, "lam-sg", "sg", ["r2_fast", "r2_hard", "r2_slow", "r2_noise", "mlp_r2_fast", "mlp_r2_hard", "rank_z", "wb_ratio", "sigreg", "pred_mse"])
display(t6b.round(3))
held = {f"weight {l}": gcfg("sg", lam=l, steps=STEPS, early_eval=EARLY, tag=f"lam-sg{l}") for l in LAMS
        if len(A.select(df, arm="sg", tag=f"lam-sg{l}", seed=SEED))}
print("what the latent is a function of (MLP from the factors to z_t, held-out R²)")
display(A.latent_determined_by(held, OUT))

**Reading stage 6**
- 6a, facing or hand (the MLP probe above all) rising as the weight falls, highest at 0: SIGReg prunes them in the shadow, and the weight is a
  lever (small but nonzero may keep the latent usable). Flat ≈ 0 at every weight: the reconstruction does not ask for them (3.5% and 1% of the
  loss), so the failure of B's shadow is the loss, not SIGReg; the frame-difference loss or another selection signal is needed.
- 6a, `shadow_rec_mse` against the weight: how much reconstruction SIGReg costs. A shadow that reconstructs the map well but not the facing, at
  every weight, is the pixel-mass bias shown directly.
- 6b, `rank_z` falling with the weight: the latent collapses (the prediction loss alone has the trivial solution), and what looks like less pruning
  at the smallest weights is emptiness. The weights with a healthy rank are the ones to read: does `facing+hand` or `inventory+position` grow at
  0.01 to 0.03 against 0.1? Then SIGReg's strength is a real lever; if the map share stays at 0.7 to 0.9 at every healthy weight, the selection is
  not the weight but the content's mass.

## 7 More seeds (optional)
Stage 2 for `EXTRA_SEEDS`, then the seed mean and spread of the tables. One seed can hide the seed-dependent availability seen for the hard sprite
in the clip world.

In [ ]:
RUN_ALL_SEEDS = False
if RUN_ALL_SEEDS:
    extra = [cfg(arm, s, steps=STEPS, early_eval=EARLY, env=env(a), tag=f"mass-a{a}")
             for s in EXTRA_SEEDS for arm in ("detached", "sg") for a in MASS_A]
    A.execute(extra, OUT, workers=WORKERS)
    df = A.collect(OUT)
    seeds = (SEED, *EXTRA_SEEDS)
    d = df[df.tag.str.startswith("mass-a") & df.seed.isin(seeds)].copy()
    d["mass %"] = d["env.a_fast"].map(MASS).round(2)
    for m in ("r2_fast", "r2_hard"):
        g = d.pivot_table(index="mass %", columns="arm", values=m, aggfunc=["mean", "std"])
        print(m, "(mean, sd over seeds)")
        display(g.round(3))
    plot_mass(sweep_tables(df, "mass-a", seeds=seeds));

## 8 Verdicts
Rules fixed above: kept = R² ≥ 0.5, lost = R² < 0.2; H2 within 0.15; H3 R² up by 0.3; H5 in 6a: the shadow's readout of facing or hand at weight 0 at least 0.3
above its value at weight 0.1 (the MLP probe where available).

In [ ]:
def crossing(s, thr=0.5):
    hit = s[s >= thr]
    return None if hit.empty else float(hit.index.min())


tabs = sweep_tables(df, "mass-a")
print("H1: smallest sprite share (% of the pixel variance) at which the readout reaches 0.5; Craftax: kept from about 1.4%, lost under 0.35%")
display(pd.DataFrame({m: {arm: crossing(tabs[m][arm]) for arm in tabs[m].columns} for m in ("r2_fast", "r2_hard", "mlp_r2_fast", "mlp_r2_hard")}))
low = [m for m in tabs["r2_fast"].index if m <= 0.6]
print(f"lost at the Craftax masses (<= 0.6%): r2_fast max {tabs['r2_fast'].loc[low].max().round(2).to_dict()}, r2_hard max {tabs['r2_hard'].loc[low].max().round(2).to_dict()}")
if {"detached", "sg"} <= set(tabs["r2_fast"].columns):
    gap = max((tabs[m]["detached"] - tabs[m]["sg"]).abs().max() for m in ("r2_fast", "r2_hard"))
    print(f"H2: largest |detached - sg| over the grid: {gap:.2f}  ->", "holds (<= 0.15)" if gap <= 0.15 else "does not hold: the prediction loss changes the selection somewhere")
try:
    gain = {k: t3.loc[(k, "frames + differences"), "r2_fast"] - t3.loc[(k, "frames"), "r2_fast"] for k in t3.index.get_level_values(0).unique()}
    print("H3: r2_fast gain of frames + differences over frames only:", {k: round(v, 2) for k, v in gain.items()}, "(prediction: >= 0.3)")
    print("    noise R² frames -> frames + differences:", {k: (round(t3.loc[(k, 'frames'), 'r2_noise'], 2), round(t3.loc[(k, 'frames + differences'), 'r2_noise'], 2)) for k in gain})
except NameError:
    print("H3: run stage 4 first")
try:
    for g_ in ("fast", "hard"):
        col = f"sh_mlp_r2_{g_}" if f"sh_mlp_r2_{g_}" in t6a and t6a[f"sh_mlp_r2_{g_}"].notna().any() else f"sh_r2_{g_}"
        lift = t6a.loc[0.0, col] - t6a.loc[0.1, col]
        print(f"H5 (6a): shadow {col}: weight 0 -> {t6a.loc[0.0, col]:.2f}, weight 0.1 -> {t6a.loc[0.1, col]:.2f}  (lift {lift:+.2f}; prediction >= +0.3)")
except NameError:
    print("H5: run stage 6 first")

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")